# Hirriririir (SegResNetDS) Thigh Segmentation — Augmented Dataset (Lambda)

Runs the MONAI **SegResNetDS** model on the 20 augmented NIfTI water volumes.
Checkpoint (~330 MB) downloads automatically.

Data: `~/our_augmented_dataset/{stem}_augmented000_water.nii.gz`
Output: `~/hirriririir_augmented_segs/{stem}_thigh_seg.nii.gz`

## 1 — Upload to Lambda
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /path/to/our_augmented_dataset/ \
  ubuntu@<YOUR-LAMBDA-IP>:~/our_augmented_dataset/
```

## 2 — Download results
```bash
rsync -avz --mkpath -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@<YOUR-LAMBDA-IP>:~/hirriririir_augmented_segs/ \
  /path/to/local/multimodal-multiethnic/augmented_segs/
```
**Terminate the instance when done.**

In [ ]:
import subprocess, sys, importlib

# NumPy 2.x breaks the system-installed torch/scipy/matplotlib on Lambda.
# Pin to <2 first, then restart the kernel before running any other cell.
_np_ver = tuple(int(x) for x in importlib.import_module('numpy').__version__.split('.')[:2])
if _np_ver >= (2, 0):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numpy<2'])
    print('NumPy downgraded to <2.  *** RESTART THE KERNEL NOW, then re-run from cell 1. ***')
else:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'monai', 'SimpleITK'])
    print('Dependencies ready')

In [ ]:
import glob, os, urllib.request
import numpy as np
import torch
import SimpleITK as sitk
from monai.networks.nets import SegResNetDS
from monai.inferers import sliding_window_inference

DATA_DIR   = os.path.expanduser('~/our_augmented_dataset')
OUTPUT_DIR = os.path.expanduser('~/hirriririir_augmented_segs')
CHECKPOINT = os.path.expanduser('~/pretrained_segmentation_muscle.pt')

TARGET_SPACING = (0.7813, 0.7813, 4.0)
ROI_SIZE       = [336, 336, 88]
DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'

LABEL_MAP = {
    1: 'Sartorius', 2: 'Rectus_Femoris', 3: 'Vastus_Lateralis',
    4: 'Vastus_Intermedius', 5: 'Vastus_Medialis', 6: 'Adductor_Magnus',
    7: 'Gracilis', 8: 'Biceps_Femoris_Long', 9: 'Semitendinosus',
    10: 'Semimembranosus', 11: 'Biceps_Femoris_Short',
}

os.makedirs(OUTPUT_DIR, exist_ok=True)

nii_files = sorted(glob.glob(os.path.join(DATA_DIR, '*_augmented*_water.nii.gz')))
print(f'Device: {DEVICE}  |  {len(nii_files)} volumes')

In [ ]:
if not os.path.exists(CHECKPOINT):
    print('Downloading checkpoint...')
    url = ('https://github.com/Hirriririir/Multimodal-Multiethnic-Thigh-Muscle-MRI-analysis'
           '/releases/download/1.0/pretrained_segmentation_muscle.pt')
    urllib.request.urlretrieve(url, CHECKPOINT)
    print(f'Done ({os.path.getsize(CHECKPOINT)//1_000_000} MB)')
else:
    print('Checkpoint present')

model = SegResNetDS(
    spatial_dims=3, in_channels=1, out_channels=12, init_filters=32,
    blocks_down=(1, 2, 2, 4, 4), dsdepth=4, norm='INSTANCE',
    resolution=TARGET_SPACING,
)
ckpt  = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
state = ckpt.get('state_dict') or ckpt.get('network_weights') or ckpt
model.load_state_dict(state, strict=False)
model = model.to(DEVICE).eval()
print('Model ready on', DEVICE)

In [ ]:
def resample_sitk(sitk_img, new_spacing, interp=sitk.sitkLinear):
    orig_sp = sitk_img.GetSpacing()
    orig_sz = sitk_img.GetSize()
    new_sz  = [int(round(orig_sz[i] * orig_sp[i] / new_spacing[i])) for i in range(3)]
    r = sitk.ResampleImageFilter()
    r.SetOutputSpacing(new_spacing); r.SetSize(new_sz)
    r.SetOutputDirection(sitk_img.GetDirection())
    r.SetOutputOrigin(sitk_img.GetOrigin())
    r.SetTransform(sitk.Transform()); r.SetDefaultPixelValue(0)
    r.SetInterpolator(interp)
    return r.Execute(sitk_img)


for nii_path in nii_files:
    basename = os.path.basename(nii_path)
    stem     = basename.replace('_water.nii.gz', '')
    out_path = os.path.join(OUTPUT_DIR, f'{stem}_thigh_seg.nii.gz')

    if os.path.exists(out_path):
        print(f'Skipping: {stem}')
        continue

    print(f'\nProcessing: {stem}')
    # Read NIfTI directly — no DICOM conversion needed
    orig = sitk.ReadImage(nii_path)
    res  = resample_sitk(orig, TARGET_SPACING)
    arr  = sitk.GetArrayFromImage(res).astype(np.float32).transpose(2, 1, 0)
    mask = arr > 0
    if mask.any():
        arr[mask] = (arr[mask] - arr[mask].mean()) / (arr[mask].std() + 1e-8)

    t = torch.tensor(arr[None, None]).float().to(DEVICE)
    with torch.no_grad():
        out = sliding_window_inference(t, roi_size=ROI_SIZE, sw_batch_size=1,
                                       predictor=model, overlap=0.5, mode='gaussian')
    logits = out[0] if isinstance(out, (list, tuple)) else out
    pred   = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy().astype(np.uint8)

    pred_sitk = sitk.GetImageFromArray(pred.transpose(2, 1, 0))
    pred_sitk.CopyInformation(res)
    pred_orig = sitk.Resample(pred_sitk, orig, sitk.Transform(), sitk.sitkNearestNeighbor, 0)
    sitk.WriteImage(pred_orig, out_path)

    pa = sitk.GetArrayFromImage(pred_orig)
    print(f'  Labels: {sorted(np.unique(pa[pa>0]).tolist())}  → {out_path}')

print('\nAll done.')

In [ ]:
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*_thigh_seg.nii.gz')))
print(f'Output files: {len(results)} / {len(nii_files)}')
if results:
    arr = sitk.GetArrayFromImage(sitk.ReadImage(results[0]))
    print(f'Sample shape: {arr.shape}  labels: {sorted(np.unique(arr[arr>0]).tolist())}')